# Notebook 1: Introduction to PyTorch
**Deep Learning · Week 1**

**PyTorch** is the deep learning library we use all semester. Everything in it is built on one object, the **tensor**: a multi-dimensional array, like a NumPy array, that can also run on a GPU. Today is about tensors only: how to create them, compute with them, change their shape, and move them between NumPy, the CPU and the GPU.

| Part | Topic |
|---|---|
| 0 | Setup |
| 1 | What is a tensor? Scalars, vectors, matrices |
| 2 | Creating tensors |
| 3 | Data types |
| 4 | Operations and matrix multiplication |
| 5 | Aggregation: min, max, mean, sum, argmax |
| 6 | Reshaping, stacking, squeezing, permuting |
| 7 | Indexing |
| 8 | Broadcasting, in NumPy and PyTorch |
| 9 | Copying tensors, and PyTorch ↔ NumPy |
| 10 | Reproducibility |
| 11 | Running on the GPU |
| ★ | Exercises |



## 0 · Setup
In Colab: **Runtime → Change runtime type → T4 GPU → Save**. Then:

In [ ]:
import torch
import numpy as np

print("Pytorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

Pytorch version: 2.11.0+cu130
GPU available: True


## 1 · What is a tensor?
A tensor is a way to represent numbers. Its **number of dimensions** (`ndim`) is the number of indices you need to reach one number.

In [ ]:
scalar = torch.tensor(7)
scalar

tensor(7)

In [ ]:
scalar.ndim # check dimensions of tensor

0

In [ ]:
scalar.item() # a one-element tensor back to a Python number

7

In [ ]:
vector = torch.tensor([7, 7, 7, 7])
vector

tensor([7, 7, 7, 7])

In [ ]:
vector[0].item()

7

In [ ]:
vector.tolist() # converts a PyTorch tensor into a standard Python list

[7, 7, 7, 7]

In [ ]:
print(f'check dimention of vector : {vector.ndim}')

check dimention of vector : 1


In [ ]:
print(f'check shape of vector : {vector.shape}')

check shape of vector : torch.Size([4])


In [ ]:
matrix = torch.tensor([[7, 8],
                       [9, 10],
                       [4, 5]])
print(matrix)

tensor([[ 7,  8],
        [ 9, 10],
        [ 4,  5]])


In [ ]:
matrix.tolist()

[[7, 8], [9, 10], [4, 5]]

In [ ]:
print(matrix.ndim)

2


In [ ]:
print(matrix.shape)

torch.Size([3, 2])


In [ ]:
tensor = torch.tensor([[[1, 2, 3],
                        [3, 6, 9],
                        [2, 4, 5]]])
print(tensor)

tensor([[[1, 2, 3],
         [3, 6, 9],
         [2, 4, 5]]])


In [ ]:
tensor.tolist()

[[[1, 2, 3], [3, 6, 9], [2, 4, 5]]]

In [ ]:
print(tensor.ndim)

3


In [ ]:
print(tensor.shape)

torch.Size([1, 3, 3])


| Name | ndim | Example shape | In this course |
|---|---|---|---|
| scalar | 0 | `()` | a loss value |
| vector | 1 | `(n,)` | one example with n features; the labels of a batch |
| matrix | 2 | `(m, n)` | a dataset: **m examples in rows, n features in columns** |
| tensor | 3+ | `(N, C, H, W)` | a batch of N images, C color channels, H × W pixels |

## 2 · Creating tensors


### 2.2 Factory functions
Most of the time you ask for a tensor of a given shape. The names are the same as in NumPy.

In [ ]:
# Creates a tensor filled with zeros of a specified size
torch.zeros(2, 3)

tensor([[0., 0., 0.],
        [0., 0., 0.]])

In [ ]:
# Creates a tensor filled with ones of a specified size.
torch.ones(2, 3)

tensor([[1., 1., 1.],
        [1., 1., 1.]])

In [ ]:
# Creates a tensor filled with a single specified scalar value.
torch.full((2, 3), 10)

tensor([[10, 10, 10],
        [10, 10, 10]])

In [ ]:
#torch.full((4,), 0.2)

In [ ]:
torch.eye(4) # identity matrix

tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]])

In [ ]:
torch.arange(0, 10)     # 0, 1, ...., 9 (the end is excluded, like range)

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [ ]:
torch.arange(0, 10, 2)   # step 2

tensor([0, 2, 4, 6, 8])

In [ ]:
torch.linspace(0, 1, 5)     # returns a one-dimensional tensor of evenly spaced values over a specified interval

tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])

**Random tensors** are how neural networks start: the weights begin with random values, and training improves them.

In [ ]:
torch.rand(4)       # uniform on [0, 1)

tensor([0.0763, 0.3207, 0.5154, 0.4191])

In [ ]:
torch.rand(2, 3)

tensor([[0.0805, 0.4553, 0.8254],
        [0.4307, 0.9690, 0.9180]])

In [ ]:
torch.randn(2, 3)     # standard normal: mean 0, std 1

tensor([[-0.0119,  0.0952,  0.5626],
        [ 1.0055,  0.6456, -1.0870]])

In [ ]:
torch.randint(0, 10, (4, 5))    # random integers from 0 to 9

tensor([[1, 6, 9, 2, 3],
        [4, 8, 6, 5, 7],
        [5, 3, 7, 8, 9],
        [3, 9, 1, 3, 5]])

In [ ]:
random_image = torch.rand(3, 224, 224)       # a random "image": 3 color channels, 224 × 224 pixels
print(f'shape of tensor : {random_image.shape}, and dimension of tensor image {random_image.ndim}')

shape of tensor : torch.Size([3, 224, 224]), and dimension of tensor image 3


### 2.3 create a new tensor with the exact same shape and data type (dtype) as an existing tensor: `*_like`



In [ ]:
x = torch.rand(2, 2, 3)
x

tensor([[[0.9218, 0.1357, 0.7707],
         [0.6259, 0.9875, 0.9742]],

        [[0.9817, 0.5300, 0.9401],
         [0.4248, 0.5326, 0.7506]]])

In [ ]:
x.shape

torch.Size([2, 2, 3])

In [ ]:
# # Manual Way
# new_tensor = torch.zeros(x.shape, dtype=x.dtype, device=x.device)
# new_tensor

In [ ]:
new_tensor = torch.zeros_like(x)
new_tensor

tensor([[[0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.]]])

In [ ]:
torch.ones_like(x)

tensor([[[1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.]]])

In [ ]:
torch.rand_like(x)

tensor([[[0.2259, 0.7548, 0.5036],
         [0.7171, 0.1065, 0.0521]],

        [[0.0460, 0.9067, 0.3972],
         [0.7977, 0.0167, 0.9440]]])

In [ ]:
torch.full_like(x, 9)

tensor([[[9., 9., 9.],
         [9., 9., 9.]],

        [[9., 9., 9.],
         [9., 9., 9.]]])

In [ ]:
print(torch.zeros_like(x).shape)
print(torch.ones_like(x).shape)
print(torch.rand_like(x).shape)
print(torch.full_like(x, 2).shape)

torch.Size([2, 2, 3])
torch.Size([2, 2, 3])
torch.Size([2, 2, 3])
torch.Size([2, 2, 3])


## 3 · Data types

Every tensor has **one** data type (`dtype`) for all its numbers. The defaults:
- integers → `torch.int64`
- decimals → **`torch.float32`**, the type neural networks use

In [ ]:
print(torch.tensor([1, 2, 3]).dtype)
print(torch.tensor([1.0, 2.0, 3.0]).dtype)

torch.int64
torch.float32


Choose the type at creation with `dtype=`, or convert later with `.to()` (or the shortcuts `.float()`, `.long()`, `.int()`):

In [ ]:
a = torch.ones((2, 3), dtype=torch.int16)
print(a)

tensor([[1, 1, 1],
        [1, 1, 1]], dtype=torch.int16)


### Dol trerm ng


In [ ]:
b = torch.rand((2, 3), dtype=torch.float64) * 20.0
print(b)

tensor([[15.1994,  5.3381, 10.3876],
        [18.5938, 14.9381,  7.8115]], dtype=torch.float64)


In [ ]:
# .to() is a multi-purpose method that can change a tensor's data type, its device, or both at the same time.

c = b.to(torch.int32)     # converting to integers truncates the decimals
print(c)

tensor([[15,  5, 10],
        [18, 14,  7]], dtype=torch.int32)


### PyTorch provides quick helper methods for casting tensors to the most commonly used data types without needing to type `torch.dtype`

In [ ]:
print(b.float().dtype)

torch.float32


In [ ]:
print(c.long().dtype)

torch.int64


In [ ]:
print(c.int().dtype)

torch.int32


**The float64 trap.** NumPy's default for decimals is `float64`, and PyTorch **keeps** that type when it converts a NumPy array. A model expects `float32`, so you get errors like *"expected scalar type Float but found Double"*. Convert with `.float()`.

In [ ]:
arr = np.array([1.0, 2.0, 3.0])
t = torch.tensor(arr)
print(arr.dtype, "->", t.dtype)
print(t.float().dtype)

float64 -> torch.float64
torch.float32


Some operations need a floating-point type. For example, `mean` refuses integers:

In [ ]:
ints = torch.arange(0, 100, 10)
try:
    ints.mean()
except RuntimeError as e:
    print("RuntimeError:", e)
print(ints.float().mean())

RuntimeError: mean(): could not infer output dtype. Input dtype must be either a floating point or complex dtype. Got: Long
tensor(45.)


**Three questions to ask when something goes wrong:** what is the tensor's **shape**? its **dtype**? its **device**? These three attributes cause most errors in deep learning code.

In [ ]:
some_tensor = torch.rand(3, 4)
print(f"shape: {some_tensor.shape} | dtype: {some_tensor.dtype} | device: {some_tensor.device}")

shape: torch.Size([3, 4]) | dtype: torch.float32 | device: cpu


## 4 · Operations

### 4.1 Element-wise arithmetic
With a number (a scalar), the operation is applied to every element. Between two tensors of the **same shape**, it is applied element by element.

In [ ]:
ones = torch.zeros(2, 2) + 1
twos = torch.ones(2, 2) * 2
threes = (torch.ones(2, 2) * 7 - 1) / 2
fours = twos ** 2
print(ones, twos, threes, fours, sep="\n")

tensor([[1., 1.],
        [1., 1.]])
tensor([[2., 2.],
        [2., 2.]])
tensor([[3., 3.],
        [3., 3.]])
tensor([[4., 4.],
        [4., 4.]])


In [ ]:
powers2 = twos ** torch.tensor([[1, 2], [3, 4]])
print(powers2)
print(threes * fours)                        # element-wise product

tensor([[ 2.,  4.],
        [ 8., 16.]])
tensor([[12., 12.],
        [12., 12.]])


In [ ]:
x = torch.tensor([-2.0, 0.0, 2.0])
print(torch.exp(x), torch.abs(x), torch.clamp(x, -1, 1), sep="\n")
print(1 / (1 + torch.exp(-x)))              # the sigmoid function

tensor([0.1353, 1.0000, 7.3891])
tensor([2., 0., 2.])
tensor([-1.,  0.,  1.])
tensor([0.1192, 0.5000, 0.8808])


### 4.2 In-place operations
`a + b` creates a **new** tensor. In-place operations are operations that directly modify the data of a tensor in memory without allocating and creating a new tensor.

In [ ]:
a = torch.ones(2, 2)
b = torch.rand(2, 2)

print("Before:")
print(a)
print(b)
print("\nAfter adding:")
print(a.add_(b))
print(a)
print(b)
print("\nAfter multiplying")
print(b.mul_(b))
print(b)

Before:
tensor([[1., 1.],
        [1., 1.]])
tensor([[0.1094, 0.2647],
        [0.8602, 0.6736]])

After adding:
tensor([[1.1094, 1.2647],
        [1.8602, 1.6736]])
tensor([[1.1094, 1.2647],
        [1.8602, 1.6736]])
tensor([[0.1094, 0.2647],
        [0.8602, 0.6736]])

After multiplying
tensor([[0.0120, 0.0701],
        [0.7399, 0.4537]])
tensor([[0.0120, 0.0701],
        [0.7399, 0.4537]])


### 4.3 Matrix multiplication
Two different products:
- `*`: element-wise, the shapes must match;
- `@` (or `torch.matmul`): the **matrix product**.

The matrix product follows two rules:
1. The **inner** dimensions must match: `(3, 2) @ (2, 4)` works, `(3, 2) @ (3, 2)` does not.
2. The result has the **outer** dimensions: `(3, 2) @ (2, 4)` → `(3, 4)`.


In [ ]:
t = torch.tensor([1, 2, 3])
print(t * t)                                 # element-wise: [1, 4, 9]
print(t @ t)                                 # dot product: 1 + 4 + 9 = 14

tensor([1, 4, 9])
tensor(14)


In [ ]:
A = torch.tensor([[1, 2], [3, 4], [5, 6]], dtype=torch.float32)      # (3, 2)
B = torch.tensor([[7, 10], [8, 11], [9, 12]], dtype=torch.float32)   # (3, 2)
try:
    A @ B
except RuntimeError as e:
    print("RuntimeError:", e)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (3x2 and 3x2)


This is **the most common error in deep learning**: a shape mismatch. Read the message, it gives both shapes. Here the fix is to **transpose** one matrix: `B.T` swaps its rows and columns, so `(3, 2)` becomes `(2, 3)`.

In [ ]:
print(A.shape, "@", B.T.shape)
print(A @ B.T)
print("result shape:", (A @ B.T).shape)

torch.Size([3, 2]) @ torch.Size([2, 3])
tensor([[ 27.,  30.,  33.],
        [ 61.,  68.,  75.],
        [ 95., 106., 117.]])
result shape: torch.Size([3, 3])


## 5 · Aggregation: min, max, mean, sum



In [ ]:
x = torch.arange(0, 100, 10).float()
print(x)
print("min:", x.min(), "| max:", x.max(), "| mean:", x.mean(), "| sum:", x.sum())
print("argmin:", x.argmin(), "| argmax:", x.argmax())      # positions, not values

tensor([ 0., 10., 20., 30., 40., 50., 60., 70., 80., 90.])
min: tensor(0.) | max: tensor(90.) | mean: tensor(45.) | sum: tensor(450.)
argmin: tensor(0) | argmax: tensor(9)


On a matrix, `dim` says **which dimension disappears**:
- `dim=0` collapses the rows → one value **per column** (per feature);
- `dim=1` collapses the columns → one value **per row** (per example).

`keepdim=True` keeps the collapsed dimension with size 1, which is useful for broadcasting (Part 8).

In [ ]:
M = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print(M.sum(dim=0))                          # per column: [5, 7, 9]
print(M.sum(dim=1))                          # per row: [6, 15]
print(M.sum(dim=1, keepdim=True).shape)      # (2, 1)
print(M.max(dim=1))                          # values and positions

tensor([5., 7., 9.])
tensor([ 6., 15.])
torch.Size([2, 1])
torch.return_types.max(
values=tensor([3., 6.]),
indices=tensor([2, 2]))


## 6 · Reshaping, stacking, squeezing, permuting

| Method | What it does |
|---|---|
| `x.reshape(shape)` | new shape, same values in the same order (the number of elements cannot change) |
| `x.view(shape)` | like `reshape`, but always shares memory with `x` |
| `torch.stack`, `torch.cat` | combine tensors: along a **new** dimension, or an **existing** one |
| `x.squeeze()` | remove dimensions of size 1 |
| `x.unsqueeze(d)` | add a dimension of size 1 at position `d` |
| `x.permute(...)` | reorder the dimensions |


In [ ]:
x = torch.arange(1.0, 10.0)
print(x, x.shape)
print(x.reshape(3, 3))
print(x.reshape(1, 9).shape, x.reshape(9, 1).shape)
print(x.reshape(3, -1).shape)                # -1: computed for you (9 / 3 = 3)

tensor([1., 2., 3., 4., 5., 6., 7., 8., 9.]) torch.Size([9])
tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]])
torch.Size([1, 9]) torch.Size([9, 1])
torch.Size([3, 3])


In [ ]:
z = x.view(3, 3)
z[0, 0] = 100                                # changing the view ...
print(x)                                     # ... changes x too: they share memory

tensor([100.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.])


In [ ]:
a, b = torch.ones(3), torch.zeros(3)
print(torch.cat([a, b]))                     # (6,)
print(torch.stack([a, b]))                   # (2, 3): new dimension in front
print(torch.stack([a, b], dim=1))            # (3, 2)

tensor([1., 1., 1., 0., 0., 0.])
tensor([[1., 1., 1.],
        [0., 0., 0.]])
tensor([[1., 0.],
        [1., 0.],
        [1., 0.]])


**`unsqueeze` for a batch of one.** Models expect a **batch**: shape (N, C, H, W). To give them a single image, add a batch dimension of size 1.


In [ ]:
img = torch.rand(3, 226, 226)
batch = img.unsqueeze(0)
print(img.shape, "->", batch.shape)

out = torch.rand(1, 20)                      # a model's output for a batch of one
print(out.shape, "->", out.squeeze(0).shape)
print(torch.rand(1, 1, 1, 10).squeeze().shape)

torch.Size([3, 226, 226]) -> torch.Size([1, 3, 226, 226])
torch.Size([1, 20]) -> torch.Size([20])
torch.Size([10])


**`permute` for images.** Image files, NumPy and Matplotlib store images as **(H, W, C)**; PyTorch layers expect **(C, H, W)**. `permute` reorders the dimensions. Using `reshape` instead gives the right **shape** but **scrambled pixels**.

In [ ]:
img_hwc = torch.rand(32, 32, 3)              # height, width, color
img_chw = img_hwc.permute(2, 0, 1)           # new order: old dim 2, then 0, then 1
print(img_hwc.shape, "->", img_chw.shape)
print("same pixel:", img_hwc[5, 7, 2].item(), img_chw[2, 5, 7].item())

wrong = img_hwc.reshape(3, 32, 32)
print("reshape gives the same image?", torch.equal(wrong, img_chw))

torch.Size([32, 32, 3]) -> torch.Size([3, 32, 32])
same pixel: 0.5386459827423096 0.5386459827423096
reshape gives the same image? False


## 7 · Indexing
Exactly like NumPy. For a matrix, **the first index is the row**.

In [ ]:
x = torch.arange(1, 10).reshape(1, 3, 3)
print(x)
print(x[0])                                  # the first (and only) block
print(x[0, 0])                               # its first row
print(x[0, 2, 2])                            # one number
print(x[:, :, 1])                            # the middle column, all rows
print(x[x > 5])                              # boolean mask: all values greater than 5

tensor([[[1, 2, 3],
         [4, 5, 6],
         [7, 8, 9]]])
tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])
tensor([1, 2, 3])
tensor(9)
tensor([[2, 5, 8]])
tensor([6, 7, 8, 9])


## 8 · Broadcasting

Operations on tensors of **different** shapes are sometimes allowed: dimensions of size 1 are stretched to match, without copying any data. This is called **broadcasting**. You may know it from NumPy; **PyTorch uses exactly the same rules**.

**The rule.** Compare the shapes **from the right**. Two dimensions are compatible if they are **equal**, or **one of them is 1**, or **one of them is missing**.

### 8.1 In NumPy

In [ ]:
X = np.zeros((5, 3))
print((X + np.array([1.0, 2.0, 3.0])).shape)            # (5, 3) + (3,): added to every row
print((X + np.arange(5.0).reshape(5, 1)).shape)         # (5, 3) + (5, 1): added to every column

col = np.arange(1.0, 4.0).reshape(3, 1)                 # (3, 1)
row = np.arange(1.0, 5.0).reshape(1, 4)                 # (1, 4)
print(col * row)                                        # (3, 4): a multiplication table

(5, 3)
(5, 3)
[[ 1.  2.  3.  4.]
 [ 2.  4.  6.  8.]
 [ 3.  6.  9. 12.]]


### 8.2 The same in PyTorch

In [ ]:
table = torch.from_numpy(col) * torch.from_numpy(row)
print(table)
print("same as NumPy:", np.array_equal(table.numpy(), col * row))

tensor([[ 1.,  2.,  3.,  4.],
        [ 2.,  4.,  6.,  8.],
        [ 3.,  6.,  9., 12.]], dtype=torch.float64)
same as NumPy: True


In [ ]:
a = torch.ones(4, 3, 2)
b = a * torch.rand(3, 2)       # last two dims identical, first one missing
c = a * torch.rand(3, 1)       # last dim is 1, middle dim identical
d = a * torch.rand(1, 2)       # last dim identical, middle dim is 1
print(b.shape, c.shape, d.shape)
print(c[0])                    # every row of the 3 × 2 block is multiplied by its own number

torch.Size([4, 3, 2]) torch.Size([4, 3, 2]) torch.Size([4, 3, 2])
tensor([[0.7607, 0.7607],
        [0.9970, 0.9970],
        [0.8461, 0.8461]])


## 9 · Copying tensors, and PyTorch ↔ NumPy

### 9.1 Assignment does not copy
`b = a` gives the same tensor a second name. To get an independent copy, use `clone()` (NumPy: `copy()`).


In [ ]:
a = torch.ones(2, 2)
b = a
a[0, 1] = 561
print(b)                                     # b changed too: same tensor

a = torch.ones(2, 2)
b = a.clone()
a[0, 1] = 561
print(b)                                     # b is unchanged

tensor([[  1., 561.],
        [  1.,   1.]])
tensor([[1., 1.],
        [1., 1.]])


### 9.2 NumPy ↔ PyTorch
| Direction | Function | Memory |
|---|---|---|
| NumPy → PyTorch | `torch.from_numpy(arr)` | **shared** with `arr` |
| NumPy → PyTorch | `torch.tensor(arr)` | copied |
| PyTorch → NumPy | `t.numpy()` | **shared** with `t` (CPU tensors only) |


In [ ]:
numpy_array = np.ones((2, 3))
pytorch_tensor = torch.from_numpy(numpy_array)
numpy_array[1, 1] = 23
print(pytorch_tensor)                        # changed too, and dtype float64 (NumPy's default)

copied = torch.tensor(numpy_array)
numpy_array[0, 0] = -1
print(copied)                                # not changed

tensor([[ 1.,  1.,  1.],
        [ 1., 23.,  1.]], dtype=torch.float64)
tensor([[ 1.,  1.,  1.],
        [ 1., 23.,  1.]], dtype=torch.float64)


## 10 · Reproducibility
Random numbers make every run different. When you want the same "random" numbers again, for example to compare two experiments fairly, set the **seed** of the random number generator.


In [ ]:
torch.manual_seed(1729)
random1 = torch.rand(2, 3)
random2 = torch.rand(2, 3)

torch.manual_seed(1729)
random3 = torch.rand(2, 3)
random4 = torch.rand(2, 3)

print(torch.equal(random1, random3), torch.equal(random2, random4))   # True True
print(torch.equal(random1, random2))                                    # False

True True
False


Setting the seed **resets** the generator: the sequence of numbers after it is always the same. Note that `random2` is different from `random1`: the seed does not repeat the same number, it repeats the same **sequence**. So the seed only helps if the code between the seed and your tensor is exactly the same.

## 11 · Running on the GPU

A GPU has thousands of small cores and its **own memory**. To compute on it, the data must be **moved** to the GPU's memory.

### 11.1 Device-agnostic code
Write code that works with and without a GPU: choose the device once, then use it everywhere.



In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

x = torch.tensor([1.0, 2.0, 3.0])            # created on the CPU
x_dev = x.to(device)                         # a copy on the device
print(x.device, "->", x_dev.device)

y_dev = torch.rand(3, device=device)         # or create it there directly

device: cuda
cpu -> cuda:0


### 11.2 All tensors must be on the same device

In [ ]:
if device == "cuda":
    try:
        x + x_dev                            # one on the CPU, one on the GPU
    except RuntimeError as e:
        print("RuntimeError:", str(e)[:120], "...")
else:
    print("No GPU in this runtime: this error only happens on a GPU.")

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! ...


### 11.3 Back to the CPU
NumPy only works on the CPU: bring the tensor back first with `.cpu()`.

In [ ]:
try:
    x_dev.numpy()
    print("On the CPU, .numpy() works directly.")
except TypeError as e:
    print("TypeError:", str(e)[:100], "...")
print(x_dev.cpu().numpy())

TypeError: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory ...
[1. 2. 3.]


---
# ★ Exercises

Do them **on your own**, in order, in the empty cells. Use the documentation. Where an expected output is shown, your output should match it (on a GPU, tensors also show `device='cuda:0'`).

## A · Warm-up

### Exercise 1
Create a random tensor with shape `(7, 7)`, then multiply it (matrix product) by another random tensor with shape `(1, 7)`. *Hint:* you will need to transpose the second tensor. Print the result and its shape.

In [ ]:
# Create two random tensors
A = torch.rand(7, 7)
B = torch.rand(1, 7)

# Matrix multiplication (check the shapes first)
result = torch.matmul(A, B.T)

print(result)
print(result.shape)

tensor([[2.2152],
        [1.6503],
        [1.7238],
        [2.0127],
        [2.4607],
        [1.9153],
        [1.9241]])
torch.Size([7, 1])


### Exercise 2
Set the random seed to `0` and do Exercise 1 again. Print the result and its shape. Expected output:
```
tensor([[1.8542],
        [1.9611],
        [2.2884],
        [3.0481],
        [1.7067],
        [2.5290],
        [1.7989]]) torch.Size([7, 1])
```

In [ ]:
# Set the seed
torch.manual_seed(0)

# Create two random tensors
A = torch.rand(7, 7)
B = torch.rand(1, 7)

# Matrix multiplication
result = torch.matmul(A, B.T)

print(result)
print(result.shape)

tensor([[1.8542],
        [1.9611],
        [2.2884],
        [3.0481],
        [1.7067],
        [2.5290],
        [1.7989]])
torch.Size([7, 1])


### Exercise 3
Is there a random seed for the **GPU** too? Find it in the documentation of [`torch.cuda`](https://pytorch.org/docs/stable/cuda.html) and set it to `1234`. In a comment, explain in one sentence why `torch.manual_seed` is usually enough.

In [ ]:
# Set the GPU seed
torch.cuda.manual_seed(1234)

# Why is torch.manual_seed usually enough?
# torch.manual_seed() sets the random seed for CPU and CUDA devices, so setting a separate GPU seed is usually unnecessary.

### Exercise 4
Set `torch.manual_seed(1234)`, create two random tensors of shape `(2, 3)` **on the CPU**, then move both to `device`. Multiply them (matrix product; adjust the shape of one of them). Print the result and its shape. Expected output (on a CPU):
```
tensor([[0.3647, 0.4709],
        [0.5184, 0.5617]]) torch.Size([2, 2])
```
*Why create them on the CPU first?* The GPU has its own random number generator: the same seed gives different numbers on the GPU.

In [ ]:
# Set the seed
torch.manual_seed(1234)

# Create two random tensors on the CPU and move them to the device
device = "cuda" if torch.cuda.is_available() else "cpu"

tensor1 = torch.rand(2, 3)
tensor2 = torch.rand(2, 3)

tensor1 = tensor1.to(device)
tensor2 = tensor2.to(device)

# Matrix multiplication
result = torch.matmul(tensor1, tensor2.T)

print(result)
print(result.shape)

tensor([[0.3647, 0.4709],
        [0.5184, 0.5617]], device='cuda:0')
torch.Size([2, 2])


### Exercise 5
Find the maximum and minimum **values** of the output of Exercise 4, then the **positions** of the maximum and minimum (`argmax`, `argmin`). Expected output:
```
tensor(0.5617) tensor(0.3647)
tensor(3) tensor(0)
```

In [ ]:
# Max and min
print(torch.max(result), torch.min(result))

# Positions of the max and min
print(torch.argmax(result), torch.argmin(result))

tensor(0.5617, device='cuda:0') tensor(0.3647, device='cuda:0')
tensor(3, device='cuda:0') tensor(0, device='cuda:0')


### Exercise 6
Set the seed to `7`, create a random tensor of shape `(1, 1, 1, 10)`, then a new tensor with all the size-1 dimensions removed, of shape `(10,)`. Print both tensors and their shapes. Expected output:
```
tensor([[[[0.5349, 0.1988, 0.6592, 0.6569, 0.2328, 0.4251, 0.2071, 0.6297,
           0.3653, 0.8513]]]]) torch.Size([1, 1, 1, 10])
tensor([0.5349, 0.1988, 0.6592, 0.6569, 0.2328, 0.4251, 0.2071, 0.6297, 0.3653,
        0.8513]) torch.Size([10])
```

In [ ]:
# Set the seed
torch.manual_seed(7)

# Create a random tensor
tensor1 = torch.rand(1, 1, 1, 10)


# Remove the size-1 dimensions
tensor2 = torch.squeeze(tensor1)

# Print both tensors and their shapes
print(tensor1, tensor1.shape)
print(tensor2, tensor2.shape)

tensor([[[[0.5349, 0.1988, 0.6592, 0.6569, 0.2328, 0.4251, 0.2071, 0.6297,
           0.3653, 0.8513]]]]) torch.Size([1, 1, 1, 10])
tensor([0.5349, 0.1988, 0.6592, 0.6569, 0.2328, 0.4251, 0.2071, 0.6297, 0.3653,
        0.8513]) torch.Size([10])


### Exercise 7
The NumPy array below holds the heights of five students, in meters. Convert it to a PyTorch tensor with dtype **`float32`**, then print the tensor's dtype and the mean height, rounded to 3 decimals. Expected output:
```
torch.float32
1.688
```

In [ ]:
heights = np.array([1.62, 1.75, 1.58, 1.80, 1.69])

# Convert to a float32 tensor
heights_tensor = torch.tensor(heights, dtype=torch.float32)

# Print its dtype and its mean (3 decimals)
print(heights_tensor.dtype)
print(round(heights_tensor.mean().item(), 3))

torch.float32
1.688


## B · Applied exercises

### Exercise 8 · From images to a data matrix
`batch` holds 8 RGB images of 32 × 32 pixels in **(N, H, W, C)** format, as an image library would give them.
1. Convert it to PyTorch's format **(N, C, H, W)**: `batch_nchw`.
2. Flatten each image into one row, to get a data matrix `X_images` of shape **(m, n) = (8, 3072)**.
3. Check that pixel (row 10, column 20, green channel) of image 4 is the same in `batch` and `batch_nchw`.

Expected output:
```
torch.Size([8, 3, 32, 32]) torch.Size([8, 3072])
tensor(True)
```

In [ ]:
torch.manual_seed(0)
batch = torch.rand(8, 32, 32, 3)

# 1. (N, H, W, C) -> (N, C, H, W)
batch_nchw = batch.permute(0, 3, 1, 2)

# 2. One row per image
X_images = batch_nchw.reshape(8, -1)

# 3. Compare one pixel (green is channel 1)
print(batch_nchw.shape, X_images.shape)
print(torch.equal(batch[3, 10, 20, 1], batch_nchw[3, 1, 10, 20]))

torch.Size([8, 3, 32, 32]) torch.Size([8, 3072])
True


### Exercise 9· Standardize the features
Each column of `data` is one feature, with a very different scale. Standardize every column: subtract its mean and divide by its standard deviation, **without loops**. Compute `mu` and `sigma` of shape (1, 4) with `keepdim=True`. Then print the mean and the standard deviation of each column of the result, rounded to 4 decimals. Expected output:
```
torch.Size([1, 4]) torch.Size([1, 4])
tensor([0., 0., 0., 0.])
tensor([1., 1., 1., 1.])
```

In [ ]:
torch.manual_seed(0)
data = torch.randn(100, 4) * torch.tensor([1.0, 10.0, 100.0, 0.1]) + 5

# Per-column mean and standard deviation, shape (1, 4)
mu = data.mean(dim=0, keepdim=True)
sigma = data.std(dim=0, keepdim=True)

# Standardize
data_standardized = (data - mu) / sigma

# Print the shapes, then the mean and std of each column (4 decimals)
# (use .abs() on the means, so that -0.0 prints as 0.0)
print(mu.shape, sigma.shape)
print(data_standardized.mean(dim=0).abs().round(decimals=4))
print(data_standardized.std(dim=0).round(decimals=4))

torch.Size([1, 4]) torch.Size([1, 4])
tensor([0., 0., 0., 0.])
tensor([1., 1., 1., 1.])
